In [10]:
import pandas as pd

# ======================================================
# CONFIG — CHANGE THESE EACH YEAR
# ======================================================
EXCEL_FILE = "PortPerformance2023.xlsx"
HISTORICAL_FILE = "Port_Data_20251112.csv"
REPORTING_YEAR = 2023
OUTPUT_FILE = "Annual_Port_Statistics_2023_UPDATE.csv"

# ======================================================
# LOAD WORKBOOK + HISTORICAL DATA
# ======================================================
xls = pd.ExcelFile(EXCEL_FILE)
df_hist = pd.read_csv(HISTORICAL_FILE)

records = []

# ======================================================
# RECONSTRUCT AGRICULTURAL COMMODITY LIST (AUTHORITATIVE)
# ======================================================
ag_commodities = set(
    df_hist[df_hist["Cargo Type"] == "TOP 6 AGRICULTURAL COMMODITIES"]
    ["Trade Type"]
    .dropna()
    .unique()
)

# ======================================================
# 1. TOTAL PORT TONNAGE — Top Ports
# ======================================================
df = xls.parse("Top Ports")
df.columns = df.columns.astype(str).str.strip().str.upper()
print("TOTAL PORT TONNAGE COLUMNS:", df.columns.tolist())
for _, r in df.iterrows():
    port = r["PORT NAME"]
    rank = r["RANK"]

    flows = {
        "TOTAL": r["GRAND TOTAL"],
        "FOREIGN": r["FOREIGN TOTAL"],
        "IMPORTS": r["IMPORTS"],
        "EXPORTS": r["EXPORTS"],
        "DOMESTIC": r["DOMESTIC"]
    }

    for trade_type, value in flows.items():
        records.append({
            "Cargo Type": "TOTAL PORT TONNAGE",
            "Port_Name": port,
            "Reporting Year": REPORTING_YEAR,
            "Trade Type": trade_type,
            "Units": "Short Tons",
            "Volume": value,
            "Port Ranking": rank
        })

# ======================================================
# 2. DRY BULK TONNAGE
# ======================================================
df = xls.parse("Dry Bulk")
df.columns = df.columns.astype(str).str.strip().str.upper()
print("DRY BULK TONNAGE COLUMNS:", df.columns.tolist())
for _, r in df.iterrows():
    port = r["PORT NAME"]
    rank =None

    flows = {
        "TOTAL": r["TOTAL"],
        "FOREIGN": r["FOREIGN"],
        "IMPORTS": r["IMPORTS"],
        "EXPORTS": r["EXPORTS"],
        "DOMESTIC": r["DOMESTIC"]
    }

    for trade_type, value in flows.items():
        records.append({
            "Cargo Type": "DRY BULK TONNAGE",
            "Port_Name": port,
            "Reporting Year": REPORTING_YEAR,
            "Trade Type": trade_type,
            "Units": "Short Tons",
            "Volume": value,
            "Port Ranking": rank
        })

# ======================================================
# 3. CONTAINER ACTIVITY (TEUs)
# ======================================================
df = xls.parse("By Port TEUs")
df.columns = df.columns.astype(str).str.strip().str.upper()
print("CONTAINER ACTIVITY COLUMNS:", df.columns.tolist())
for _, r in df.iterrows():
    port = r["PORT NAME"]
    rank = None

    flows = {
        "TOTAL": r.get("TOTAL"),
        "IMPORTS": r.get("IMPORTS"),
        "EXPORTS": r.get("EXPORTS"),
        "EMPTY": r.get("EMPTY")
    }

    for trade_type, value in flows.items():
        if pd.notna(value):
            records.append({
                "Cargo Type": "CONTAINER ACTIVITY",
                "Port_Name": port,
                "Reporting Year": REPORTING_YEAR,
                "Trade Type": trade_type,
                "Units": "TEUs",
                "Volume": value,
                "Port Ranking": None
            })

# ======================================================
# 4. TOP 5 COMMODITIES
# ======================================================
df = xls.parse("Ports by Commodity")
df.columns = df.columns.astype(str).str.strip().str.upper()
print("TOP 5 COMMODITIES COLUMNS:", df.columns.tolist())
for _, r in df.iterrows():
    records.append({
        "Cargo Type": "TOP 5 COMMODITIES",
        "Port_Name": r["PORT NAME"],
        "Reporting Year": REPORTING_YEAR,
        "Trade Type": r["COMMODITY NAME"],
        "Units": "Short Tons",
        "Volume": r["TOTAL"],
        "Port Ranking": None
    })

# ======================================================
# 5. DERIVED TOP 6 AGRICULTURAL COMMODITIES
# ======================================================
for port, grp in df.groupby("PORT NAME"):

    ag_grp = grp[grp["COMMODITY NAME"].isin(ag_commodities)]

    if ag_grp.empty:
        continue

    top6 = (
        ag_grp
        .sort_values("TOTAL", ascending=False)
        .head(6)
    )

    for rank, (_, r) in enumerate(top6.iterrows(), start=1):
        records.append({
            "Cargo Type": "TOP 6 AGRICULTURAL COMMODITIES",
            "Port_Name": port,
            "Reporting Year": REPORTING_YEAR,
            "Trade Type": r["COMMODITY NAME"],
            "Units": "Short Tons",
            "Volume": r["TOTAL"],
            "Port Ranking": rank
        })

# ======================================================
# OUTPUT
# ======================================================
df_out = pd.DataFrame(records)
df_out.to_csv(OUTPUT_FILE, index=False)

print(f"✅ Wrote {len(df_out)} records to {OUTPUT_FILE}")


TOTAL PORT TONNAGE COLUMNS: ['RANK', 'TYPE', 'PORT NAME', 'GRAND TOTAL', 'FOREIGN TOTAL', 'IMPORTS', 'EXPORTS', 'DOMESTIC']
DRY BULK TONNAGE COLUMNS: ['PORT', 'PORT NAME', 'DOMESTIC', 'EXPORTS', 'IMPORTS', 'FOREIGN', 'TOTAL']
CONTAINER ACTIVITY COLUMNS: ['PORT CODE', 'PORT NAME', 'STATE', 'DOMESTIC', 'UNNAMED: 4', 'UNNAMED: 5', 'UNNAMED: 6', 'UNNAMED: 7', 'FOREIGN', 'UNNAMED: 9', 'UNNAMED: 10', 'GRAND']
TOP 5 COMMODITIES COLUMNS: ['TYPE', 'PORT', 'PORT NAME', 'COMMODITY GROUP', 'COMMODITY NAME', 'DOMESTIC', 'FOREIGN', 'IMPORTS', 'EXPORTS', 'TOTAL']
✅ Wrote 11285 records to Annual_Port_Statistics_2023_UPDATE.csv


In [11]:
df.head()

,TYPE,PORT,PORT NAME,COMMODITY GROUP,COMMODITY NAME,DOMESTIC,FOREIGN,IMPORTS,EXPORTS,TOTAL
0,Internal,2359,"Aberdeen, MS",2211,Gasoline,137352,0,0,0,137352
1,Internal,2359,"Aberdeen, MS",2330,Distillate Fuel Oil,110527,0,0,0,110527
2,Internal,2359,"Aberdeen, MS",3220,Alcohols,17198,0,0,0,17198
3,Great Lakes,3612,"Alabaster Township, MI",4323,Gypsum,212416,0,0,0,212416
4,Coastal,505,"Albany Port District, NY",2211,Gasoline,2068708,0,0,0,2068708
